# CNN training -- price movement classification

This notebook trains `Conv1DClassifier` (see `analysis/models/cnn.py`) to classify
future price movement from a rolling window of daily features plus a symbol
embedding (see `analysis/datasets/stocks.py`).

Data is split by time, never randomly, so the model is always evaluated on
data strictly *after* what it trained on:

- **train**: 2022-01-01 -- 2024-06-01
- **validation**: 2024-06-01 -- 2025-01-01, used for early stopping and to pick
  the best config out of the sweep
- **test**: 2025-01-01 -- 2026-01-01, held out untouched until the very end,
  used only once for a final unbiased score of whichever config wins the sweep

##### Imports

In [1]:
import sys

sys.path.append("../..")

from datetime import date

import matplotlib.pyplot as plt
import torch

from analysis.datasets.stocks import StocksDataset
from analysis.models.cnn import Conv1DConfig, Conv1DClassifier
from analysis.training import TrainingConfig, sweep, evaluate


##### Set up

In [2]:
DATAPLATFORM_ROOT = "../../dataplatform"

TRAIN_START, TRAIN_END = date(2022, 1, 1), date(2024, 6, 1)
VAL_START, VAL_END = date(2024, 6, 1), date(2025, 1, 1)
TEST_START, TEST_END = date(2025, 1, 1), date(2026, 1, 1)

SERIES_LENGTH = 30
LOOKAHEAD_STEPS = 5
THRESHOLDS = [0.01, 0.03] # <-3% | -3,-1% | -1,1% | 1,3% | >3%

FEATURE_LIST = [
    "open",
    "open_rolling_1w",
    "log_return_1d",
    "log_return_1w",
    "volatility_1m",
    "sharpe_1m",
    "rsi"
]



In [3]:
train_ds = StocksDataset(
    FEATURE_LIST, TRAIN_START, TRAIN_END,
    series_length=SERIES_LENGTH, lookahead_steps=LOOKAHEAD_STEPS, thresholds=THRESHOLDS,
    dataplatform_root=DATAPLATFORM_ROOT,
)
val_ds = StocksDataset(
    FEATURE_LIST, VAL_START, VAL_END,
    series_length=SERIES_LENGTH, lookahead_steps=LOOKAHEAD_STEPS, thresholds=THRESHOLDS,
    dataplatform_root=DATAPLATFORM_ROOT,
)
test_ds = StocksDataset(
    FEATURE_LIST, TEST_START, TEST_END,
    series_length=SERIES_LENGTH, lookahead_steps=LOOKAHEAD_STEPS, thresholds=THRESHOLDS,
    dataplatform_root=DATAPLATFORM_ROOT,
)

print(f"train: {len(train_ds)} samples, validation: {len(val_ds)} samples, test: {len(test_ds)} samples")


[22:41:26] Loading from disk data model stocks_daily
[22:41:26] gold/stocks_daily scan plan ready from disk
[22:41:26] Loading from disk data model symbol_embeddings
[22:41:26] silver/symbol_embeddings scan plan ready from disk


/home/danie/codice/zecca/analysis/notebooks/../../analysis/datasets/stocks.py:92: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  (c for c in df.columns if _EMBEDDING_COL.match(c)), key=lambda c: int(c[1:])
/home/danie/codice/zecca/analysis/notebooks/../../analysis/datasets/stocks.py:97: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  for c in df.columns
/home/danie/codice/zecca/analysis/notebooks/../../analysis/datasets/stocks.py:124: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  .collect()


[22:41:40] Loading from disk data model stocks_daily
[22:41:40] gold/stocks_daily scan plan ready from disk
[22:41:40] Loading from disk data model symbol_embeddings
[22:41:40] silver/symbol_embeddings scan plan ready from disk
[22:41:43] Loading from disk data model stocks_daily
[22:41:43] gold/stocks_daily scan plan ready from disk
[22:41:43] Loading from disk data model symbol_embeddings
[22:41:43] silver/symbol_embeddings scan plan ready from disk
train: 3768704 samples, validation: 805100 samples, test: 1482583 samples


#### Hyperparameter tuning

In [4]:
BASE_NETWORK_CONFIG = Conv1DConfig(
    num_classes=int(train_ds.labels.max().item()) + 1,
    num_features=train_ds.series.shape[1],
    symbol_embedding_dim=train_ds.embedding.shape[1],
)
BASE_TRAIN_CONFIG = TrainingConfig(
    learning_rate=1e-3,
    batch_size=128,
    num_epochs=30,
    patience=5,
)

SWEEP_CONFIGS = [
    {},  # baseline, unmodified
    {"learning_rate": 3e-4},
    {"cnn_width": 128, "num_blocks": 4},
    {"optimizer": torch.optim.SGD, "learning_rate": 1e-2, "optimizer_kwargs": {"momentum": 0.9}},
    {"scheduler": torch.optim.lr_scheduler.ReduceLROnPlateau, "scheduler_kwargs": {"patience": 2}},
]

results = sweep(
    SWEEP_CONFIGS, Conv1DClassifier, BASE_NETWORK_CONFIG, BASE_TRAIN_CONFIG,
    train_ds, val_ds,
)


Running on device cuda
Epoch 0/30: train_loss=1.4832109549412953 val_loss=1.5498442891884585
Epoch 1/30: train_loss=1.4604082986379947 val_loss=1.5554196390515158
Epoch 2/30: train_loss=1.449766397152291 val_loss=1.5637222955313073
Epoch 3/30: train_loss=1.443842700531604 val_loss=1.6675173779662804
Epoch 4/30: train_loss=1.439896696130409 val_loss=1.8270054712853452
Epoch 5/30: train_loss=1.437050117486133 val_loss=1.9380160053903723
Running on device cuda
Epoch 0/30: train_loss=1.4776978651652843 val_loss=1.5310934724696392
Epoch 1/30: train_loss=1.448125230748988 val_loss=1.541760263342307


KeyboardInterrupt: 

#### Compare results

In [ ]:
for override, r in zip(SWEEP_CONFIGS, results):
    print(f"{override!r:70s} -> best_val_loss={r['result'].best_val_loss:.4f}")


In [ ]:
for override, r in zip(SWEEP_CONFIGS, results):
    plt.plot(r["result"].val_losses, label=str(override) or "baseline")
plt.xlabel("epoch")
plt.ylabel("validation loss")
plt.legend(fontsize=7)
plt.show()


## Final held-out test evaluation

Pick whichever sweep entry had the lowest validation loss and score it on the
test set -- the one split no model or hyperparameter choice above has ever
seen.

In [ ]:
best = min(results, key=lambda r: r["result"].best_val_loss)
test_loss = evaluate(best["model"], test_ds, best["train_config"])

print("best config overrides:", best["overrides"])
print("validation loss:", best["result"].best_val_loss)
print("test loss:", test_loss)
